# 🚨 FloodShield AI — Model 5 & 6: Risk Score + Recommendation AI
**Model 5:** Composite District Risk Score (Low / Medium / High / Critical)
**Model 6:** Resource Recommendation (ORS, Medicine, Shelter, Emergency Kit)

### এই দুইটা কেন একসাথে একটা notebook-এ?
এই দুইটা heavy ML training না — মূলত **Model 1 ও Model 2-এর output মিলিয়ে composite scoring + rule-based decision logic**। তাই আলাদা training দরকার নাই, একটা notebook-এই দুইটা করে ফেলা যায়।

### কীভাবে চালাবে:
1. Drive-এ থাকা `disease_prediction_dataset_fixed.xlsx` ও `hospital_resources_csv.xls` ব্যবহার হবে
2. Model 1 ও Model 2-এর saved `.pkl` file গুলোও লাগবে (already `FloodShieldAI/models/`-এ আছে)
3. Runtime → Run all


## 1️⃣ Setup

In [6]:
!pip install pandas openpyxl joblib scikit-learn -q

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2️⃣ Load Datasets + Trained Models

In [7]:
import pandas as pd
import numpy as np
import joblib

BASE = '/content/drive/MyDrive/FloodShieldAI'

dp = pd.read_excel(f'{BASE}/data/disease_prediction_dataset_fixed.xlsx')
hosp = pd.read_csv(f'{BASE}/data/hospital_resources_csv.xls')

# Load Model 2 (disease outbreak) artifacts
outbreak_model = joblib.load(f'{BASE}/models/disease_outbreak_model.pkl')
outbreak_le = joblib.load(f'{BASE}/models/outbreak_risk_label_encoder.pkl')
outbreak_features = joblib.load(f'{BASE}/models/outbreak_model_features.pkl')

print("Disease data:", dp.shape)
print("Hospital data:", hosp.shape)
print("Outbreak model loaded:", type(outbreak_model))


Disease data: (30000, 24)
Hospital data: (500, 12)
Outbreak model loaded: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


## 3️⃣ Model 5 — Composite Risk Score
**Logic:**
- Disease Outbreak Risk (Model 2 output): Medium=1, High=2, Critical=3
- Flood Severity (from flood_depth_cm, রাখা হয়েছে input হিসেবে, leakage না — এটা current observed condition, future target না): None=0, Minor=1, Moderate=2, Severe=3
- **Composite Score = weighted average** (disease 60% + flood 40%, কারণ disease outbreak-ই মূল target এই platform-এর)
- Final category: Low (0-1) / Medium (1-2) / High (2-2.5) / Critical (2.5-3)


In [8]:
def flood_severity_score(depth_cm):
    if depth_cm < 10: return 0    # None
    elif depth_cm < 40: return 1  # Minor
    elif depth_cm < 80: return 2  # Moderate
    else: return 3                # Severe

def compute_risk_score(row):
    X_row = row[outbreak_features].to_frame().T
    probs = outbreak_model.predict_proba(X_row)[0]
    outbreak_class = outbreak_le.classes_[probs.argmax()]
    outbreak_map = {'Medium':1, 'High':2, 'Critical':3}
    outbreak_score = outbreak_map.get(outbreak_class, 1)

    flood_score = flood_severity_score(row['flood_depth_cm'])

    composite = 0.6*outbreak_score + 0.4*flood_score

    if composite < 1:
        level = 'Low'
    elif composite < 2:
        level = 'Medium'
    elif composite < 2.5:
        level = 'High'
    else:
        level = 'Critical'

    return pd.Series({
        'predicted_outbreak_risk': outbreak_class,
        'flood_severity_score': flood_score,
        'composite_score': round(composite, 2),
        'final_risk_level': level
    })

# Test on a sample of 200 rows (full 30,000 would be slow row-by-row; for production this becomes a batch API call)
sample = dp.sample(200, random_state=42).copy()

# prepare season one-hot to match outbreak_features
season_dummies = pd.get_dummies(sample['season'], prefix='season')
sample_full = pd.concat([sample, season_dummies], axis=1)
for col in outbreak_features:
    if col not in sample_full.columns:
        sample_full[col] = 0

risk_results = sample_full.apply(compute_risk_score, axis=1)
sample_out = pd.concat([sample[['district','date','season']], risk_results], axis=1)
print(sample_out['final_risk_level'].value_counts())
sample_out.head(10)


final_risk_level
Medium      139
High         23
Low          20
Critical     18
Name: count, dtype: int64


,district,date,season,predicted_outbreak_risk,flood_severity_score,composite_score,final_risk_level
2308,Munshiganj,2024-11-12,Post-Monsoon,High,0,1.2,Medium
22404,Bogra,2022-05-03,Summer,High,1,1.6,Medium
23397,Chandpur,2025-05-31,Summer,High,1,1.6,Medium
25058,Dinajpur,2022-12-19,Winter,High,0,1.2,Medium
2664,Sylhet,2024-12-31,Winter,High,1,1.6,Medium
8511,Brahmanbaria,2024-06-08,Monsoon,High,2,2.0,High
5148,Bogra,2024-06-19,Monsoon,High,1,1.6,Medium
7790,Shariatpur,2023-06-23,Monsoon,High,1,1.6,Medium
11311,Moulvibazar,2025-06-10,Monsoon,Critical,1,2.2,High
19043,Gaibandha,2022-02-21,Winter,High,1,1.6,Medium


## 4️⃣ Model 5 — Reusable Function (single input, for API)

In [9]:
def get_risk_score(rainfall_mm, temperature_c, humidity_percent, river_level_m, flood_depth_cm,
                    population_density, sanitation_score, water_quality_index,
                    previous_week_cases, previous_two_week_cases, season='Monsoon'):
    input_dict = {
        'rainfall_mm': rainfall_mm, 'temperature_c': temperature_c,
        'humidity_percent': humidity_percent, 'river_level_m': river_level_m,
        'flood_depth_cm': flood_depth_cm, 'population_density': population_density,
        'sanitation_score': sanitation_score, 'water_quality_index': water_quality_index,
        'previous_week_cases': previous_week_cases, 'previous_two_week_cases': previous_two_week_cases
    }
    season_cols = [c for c in outbreak_features if c.startswith('season_')]
    for col in season_cols:
        season_name = col.replace('season_', '')
        input_dict[col] = 1 if season_name == season else 0

    input_df = pd.DataFrame([input_dict])[outbreak_features]
    probs = outbreak_model.predict_proba(input_df)[0]
    outbreak_class = outbreak_le.classes_[probs.argmax()]
    outbreak_map = {'Medium':1, 'High':2, 'Critical':3}
    outbreak_score = outbreak_map.get(outbreak_class, 1)

    flood_score = flood_severity_score(flood_depth_cm)
    composite = 0.6*outbreak_score + 0.4*flood_score

    if composite < 1: level = 'Low'
    elif composite < 2: level = 'Medium'
    elif composite < 2.5: level = 'High'
    else: level = 'Critical'

    return {
        'predicted_outbreak_risk': outbreak_class,
        'flood_severity': ['None','Minor','Moderate','Severe'][flood_score],
        'composite_score': round(composite, 2),
        'final_risk_level': level
    }

# Example
print(get_risk_score(rainfall_mm=220, temperature_c=29, humidity_percent=87, river_level_m=7.5,
                      flood_depth_cm=55, population_density=2100, sanitation_score=0.35,
                      water_quality_index=50, previous_week_cases=210, previous_two_week_cases=190,
                      season='Monsoon'))


{'predicted_outbreak_risk': 'Critical', 'flood_severity': 'Moderate', 'composite_score': 2.6, 'final_risk_level': 'Critical'}


## 5️⃣ Model 6 — Recommendation AI (Rule-based Resource Recommendation)
**Logic:** District-এর risk level অনুযায়ী + সেই district-এর hospital stock (ORS/IV/Antibiotics) দেখে recommend করবে।

| Risk Level | Recommendation Trigger |
|---|---|
| Critical | ORS + IV Saline + Antibiotics + Emergency Medical Team + Shelter |
| High | ORS + IV Saline + stock alert যদি কম থাকে |
| Medium | Monitor + ORS stock alert যদি কম থাকে |
| Low | Routine monitoring |


In [10]:
# Aggregate hospital stock per district (district-এ multiple hospital থাকতে পারে)
hosp_district = hosp.groupby('district').agg({
    'beds':'sum','doctors':'sum','nurses':'sum',
    'ors_stock':'sum','iv_saline_stock':'sum','antibiotics_stock':'sum',
    'occupancy_percent':'mean'
}).reset_index()

# thresholds for "low stock" (based on dataset distribution)
ORS_LOW_THRESHOLD = hosp_district['ors_stock'].quantile(0.25)
IV_LOW_THRESHOLD = hosp_district['iv_saline_stock'].quantile(0.25)
ANTIBIOTIC_LOW_THRESHOLD = hosp_district['antibiotics_stock'].quantile(0.25)

print(f"ORS low threshold: {ORS_LOW_THRESHOLD:.0f}")
print(f"IV Saline low threshold: {IV_LOW_THRESHOLD:.0f}")
print(f"Antibiotics low threshold: {ANTIBIOTIC_LOW_THRESHOLD:.0f}")

def recommend_resources(district_name, risk_level):
    row = hosp_district[hosp_district['district']==district_name]
    if row.empty:
        return {"error": f"No hospital data found for {district_name}"}
    row = row.iloc[0]

    recommendations = []

    if risk_level == 'Critical':
        recommendations += ["Deploy Emergency Medical Team", "Prepare Emergency Shelter",
                             "Distribute Emergency Kits (household level)"]
        recommendations.append("URGENT: ORS restock needed" if row['ors_stock'] < ORS_LOW_THRESHOLD else "ORS stock adequate")
        recommendations.append("URGENT: IV Saline restock needed" if row['iv_saline_stock'] < IV_LOW_THRESHOLD else "IV Saline stock adequate")
        recommendations.append("URGENT: Antibiotics restock needed" if row['antibiotics_stock'] < ANTIBIOTIC_LOW_THRESHOLD else "Antibiotics stock adequate")

    elif risk_level == 'High':
        recommendations.append("Increase ORS supply" if row['ors_stock'] < ORS_LOW_THRESHOLD else "ORS stock adequate")
        recommendations.append("Increase IV Saline supply" if row['iv_saline_stock'] < IV_LOW_THRESHOLD else "IV Saline stock adequate")
        recommendations.append("Alert local health workers for increased monitoring")

    elif risk_level == 'Medium':
        recommendations.append("Monitor ORS stock" if row['ors_stock'] < ORS_LOW_THRESHOLD else "No immediate action needed")
        recommendations.append("Routine health advisory to community")

    else:  # Low
        recommendations.append("Routine monitoring, no special action needed")

    return {
        'district': district_name,
        'risk_level': risk_level,
        'hospital_beds_available': int(row['beds']),
        'occupancy_percent': round(row['occupancy_percent'],1),
        'recommendations': recommendations
    }

# Example
print(recommend_resources('Munshiganj', 'Critical'))


ORS low threshold: 50853
IV Saline low threshold: 14745
Antibiotics low threshold: 10091
{'district': 'Munshiganj', 'risk_level': 'Critical', 'hospital_beds_available': 1566, 'occupancy_percent': np.float64(55.4), 'recommendations': ['Deploy Emergency Medical Team', 'Prepare Emergency Shelter', 'Distribute Emergency Kits (household level)', 'ORS stock adequate', 'IV Saline stock adequate', 'Antibiotics stock adequate']}


## 6️⃣ Combined End-to-End Function (Model 5 + Model 6 together)
এটাই backend API-র `/predict/risk` এবং `/recommend` endpoint-এর core logic হবে।


In [11]:
def full_district_assessment(district_name, rainfall_mm, temperature_c, humidity_percent,
                              river_level_m, flood_depth_cm, population_density, sanitation_score,
                              water_quality_index, previous_week_cases, previous_two_week_cases,
                              season='Monsoon'):
    risk = get_risk_score(rainfall_mm, temperature_c, humidity_percent, river_level_m, flood_depth_cm,
                           population_density, sanitation_score, water_quality_index,
                           previous_week_cases, previous_two_week_cases, season)
    reco = recommend_resources(district_name, risk['final_risk_level'])

    return {
        'district': district_name,
        'risk_assessment': risk,
        'recommendation': reco
    }

# Example: full pipeline test
result = full_district_assessment(
    district_name='Munshiganj', rainfall_mm=220, temperature_c=29, humidity_percent=87,
    river_level_m=7.5, flood_depth_cm=55, population_density=2100, sanitation_score=0.35,
    water_quality_index=50, previous_week_cases=210, previous_two_week_cases=190, season='Monsoon'
)
import json
print(json.dumps(result, indent=2))


{
  "district": "Munshiganj",
  "risk_assessment": {
    "predicted_outbreak_risk": "Critical",
    "flood_severity": "Moderate",
    "composite_score": 2.6,
    "final_risk_level": "Critical"
  },
  "recommendation": {
    "district": "Munshiganj",
    "risk_level": "Critical",
    "hospital_beds_available": 1566,
    "occupancy_percent": 55.4,
    "recommendations": [
      "Deploy Emergency Medical Team",
      "Prepare Emergency Shelter",
      "Distribute Emergency Kits (household level)",
      "ORS stock adequate",
      "IV Saline stock adequate",
      "Antibiotics stock adequate"
    ]
  }
}


## 7️⃣ Save Everything (Thresholds + Hospital Aggregates)

In [12]:
import os

SAVE_DIR = f'{BASE}/models'
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(hosp_district, f'{SAVE_DIR}/hospital_district_aggregates.pkl')
joblib.dump({
    'ors': ORS_LOW_THRESHOLD, 'iv': IV_LOW_THRESHOLD, 'antibiotics': ANTIBIOTIC_LOW_THRESHOLD
}, f'{SAVE_DIR}/resource_thresholds.pkl')

print(f"✅ Saved to: {SAVE_DIR}")


✅ Saved to: /content/drive/MyDrive/FloodShieldAI/models


---
## ✅ Model 5 ও Model 6 — কী তৈরি হলো
- **Model 5:** Model 2-এর outbreak prediction + flood severity মিলিয়ে composite risk score (Low/Medium/High/Critical)
- **Model 6:** Risk level + real hospital stock data মিলিয়ে rule-based recommendation (ORS/IV/Antibiotics/Emergency Team/Shelter)

### সৎ Note
এই দুইটা "trained ML model" না, বরং **domain-logic-driven scoring ও rule engine** — যেটা বাস্তবেই এই ধরনের decision-support system-এ standard practice (পুরোপুরি black-box ML দিয়ে recommend করা ঝুঁকিপূর্ণ, rule-based হলে explainable ও audit করা যায়, যেটা government/হাসপাতাল client-দের কাছে বেশি বিশ্বাসযোগ্য)।

## পরের ধাপ
Backend FastAPI-তে এই `full_district_assessment()` ফাংশনটাই মূলত `/predict/risk` endpoint হবে।
